# 第 08 課 - 多代理人設計模式


## 安裝與設定


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 為什麼要使用多代理系統？

現實世界的任務像是行程規劃，涉及許多不同的專業知識——物流、本地知識、預算等。單一代理試圖處理所有事情很快就會變得難以掌控。

多代理系統透過<strong>專業分工</strong>來解決這個問題：每個代理專注於一個專業領域，產出比通才更高質量的結果。它們也提升了<strong>可擴展性</strong>——你可以新增代理（例如航班專家、餐廳評論家），而不必重寫現有的工作流程。代理們透過一個結構化的管線協同工作，將上下文從一個代理傳遞到下一個代理。


## 創建專門代理人


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 建立一個順序工作流程

`WorkflowBuilder` 讓你將代理人連接成一個有向圖。這裡我們建立一個簡單的兩步驟管線：**TravelPlanner** 擬定行程，然後 **TravelConcierge** 審查並加以完善。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 向工作流程中添加更多代理

多代理模式最大的優點之一就是擴展起來非常容易。以下我們新增一個 **BudgetReviewer** 代理，負責檢查計畫是否符合旅客的預算，標示可能超出預算的項目，並建議省錢的替代方案。現在工作流程會依序執行三個代理：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 總結

在本課程中，您學到了如何：

1. <strong>創建專門的代理</strong> — 每個代理都有明確的角色（規劃、禮賓、預算審核）。
2. **使用 `WorkflowBuilder` 和 `add_edge` 將代理連接成順序工作流程**。
3. <strong>串流輸出</strong> 從多代理管線，並追蹤哪個代理在發言。
4. <strong>擴展工作流程</strong>，透過新增代理到鏈中而不修改現有代理。

多代理設計模式保持每個代理簡單，同時產出比單一代理更豐富、經過更仔細審核的結果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
